# 🦴 Segmentación de Columna Vertebral con YOLOv8n-seg
## Detección de Escoliosis mediante estimación del Ángulo de Cobb

**Proyecto Académico — Visión por Computador**

---

### Diferencias respecto a U-Net

| Aspecto | U-Net (original) | YOLOv8n-seg (nuevo) |
|---|---|---|
| Arquitectura | Encoder-Decoder semántico | One-stage detector + máscara por instancia |
| Salida | Mapa de píxeles (H×W×C) | Bounding boxes + máscaras de instancia |
| Segmentación | Semántica (1 clase por píxel) | Por instancia (cada vértebra como objeto separado) |
| Formato de etiquetas | Máscara PNG multiclase | YOLO `.txt` con polígonos normalizados |
| Entrenamiento | Loop manual PyTorch | `model.train()` nativo de Ultralytics |
| Velocidad inferencia | Lenta (pase completo) | Muy rápida (~6ms/imagen en CPU) |
| Ventaja clínica | Segmentación densa exacta | Detecta y numera vértebras individualmente |

### Pipeline completo
```
Máscaras PNG multiclase
        ↓
Conversión a polígonos YOLO (.txt)
        ↓
Fine-tuning YOLOv8n-seg (pesos preentrenados COCO)
        ↓
Inferencia → máscara por instancia vertebral
        ↓
PCA por vértebra → ángulo de orientación
        ↓
Estimación del ángulo de Cobb
```

## 📦 Celda 1: Instalación de dependencias

In [1]:
# Instalar Ultralytics (incluye YOLOv8) y dependencias
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'ultralytics', 'opencv-python', 'matplotlib',
                       'scikit-learn', 'pandas', 'numpy', '-q'])

from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Ultralytics instalado correctamente')
print(f'Dispositivo: {device}')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/fedys/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics instalado correctamente
Dispositivo: cpu


## 📂 Celda 2: Carga y división del dataset

Mismo esquema que U-Net: 249 imágenes (71 Normal / 178 Escoliosis), split 70/15/15 estratificado.

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

image_root = 'images'
mask_root  = 'labels_clean'

data = []

for label_name in ['Normal', 'Scoliosis']:
    image_folder = os.path.join(image_root, label_name)

    for file in os.listdir(image_folder):
        if not file.endswith('.jpg'):
            continue

        base_name = os.path.splitext(file)[0]
        mask_name = f'LabelMulti_{base_name}.png'

        img_path  = os.path.join(image_folder, file)
        mask_path = os.path.join(mask_root, mask_name)

        if not os.path.exists(mask_path):
            print('Mask no encontrada:', mask_name)
            continue

        label = 0 if label_name == 'Normal' else 1
        data.append({'image': img_path, 'mask': mask_path, 'label': label})

df = pd.DataFrame(data)
print(df.head())
print('Total:', len(df))
print(df['label'].value_counts())

# División estratificada — mismas proporciones que U-Net
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

                    image                              mask  label
0  images/Normal/N_26.jpg  labels_clean/LabelMulti_N_26.png      0
1  images/Normal/N_32.jpg  labels_clean/LabelMulti_N_32.png      0
2  images/Normal/N_33.jpg  labels_clean/LabelMulti_N_33.png      0
3  images/Normal/N_27.jpg  labels_clean/LabelMulti_N_27.png      0
4  images/Normal/N_31.jpg  labels_clean/LabelMulti_N_31.png      0
Total: 249
label
1    178
0     71
Name: count, dtype: int64
Train: 174  |  Val: 37  |  Test: 38


## 🔄 Celda 3: Conversión de máscaras PNG → formato YOLO

### ¿Por qué es necesaria esta conversión?

YOLOv8-seg **no acepta máscaras PNG** como U-Net. Necesita etiquetas en formato `.txt`
donde cada línea representa **una instancia** con su polígono normalizado:

```
class_id  x1 y1  x2 y2  x3 y3 ... xn yn
```

Esto refleja la diferencia fundamental entre segmentación **semántica** (U-Net: un color
por clase en toda la imagen) e **instancia** (YOLO: cada vértebra es un objeto separado
con su propio contorno).

### Mapeo de clases
Las 36 clases de la máscara (0=fondo, 1-35=vértebras) se mapean a 24 clases YOLO
(C1-C7, T1-T12, L1-L5) más el fondo que se ignora.

In [3]:
import cv2
import numpy as np
from pathlib import Path

# ── Nombres de clases vertebrales ────────────────────────────────────
# Las clases 1-24 de la máscara PNG corresponden a las vértebras.
# Clase 0 = fondo (se omite). Clases 25-35 = etiquetas adicionales
# que también se omiten si no corresponden a vértebras específicas.
CLASS_NAMES = [
    'C1','C2','C3','C4','C5','C6','C7',          # Cervicales  (IDs 1-7)
    'T1','T2','T3','T4','T5','T6',                # Torácicas   (IDs 8-13)
    'T7','T8','T9','T10','T11','T12',             # Torácicas   (IDs 14-19)
    'L1','L2','L3','L4','L5',                     # Lumbares    (IDs 20-24)
]
# pixel_value → yolo_class_id  (0-indexed, pixel 0=BG ignorado)
PIXEL_TO_YOLO = {pv: pv - 1 for pv in range(1, len(CLASS_NAMES) + 1)}
N_CLASSES = len(CLASS_NAMES)   # 24


def mask_png_a_yolo_txt(mask_path: str, txt_path: str,
                         min_area: int = 100) -> int:
    """
    Convierte una máscara PNG multiclase al formato de etiquetas YOLO-seg.

    Para cada valor de píxel distinto de 0 (cada vértebra):
      1. Binariza la máscara para esa clase.
      2. Encuentra contornos (cv2.findContours).
      3. Toma el contorno más grande (la región principal).
      4. Normaliza las coordenadas al rango [0, 1].
      5. Escribe una línea: class_id x1 y1 x2 y2 ... xn yn

    Parámetros
    ----------
    mask_path : Ruta a la máscara PNG de entrada.
    txt_path  : Ruta al archivo .txt de salida.
    min_area  : Área mínima de contorno para no ignorar ruido.

    Retorna
    -------
    int : Número de instancias escritas en el .txt.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return 0

    H, W = mask.shape
    lines = []

    for pixel_val, yolo_id in PIXEL_TO_YOLO.items():
        # Binarizar: solo los píxeles de esta vértebra
        binary = (mask == pixel_val).astype(np.uint8) * 255

        if binary.sum() == 0:
            continue   # esta vértebra no aparece en la imagen

        # Encontrar contornos externos
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue

        # Tomar el contorno más grande (evita fragmentos de ruido)
        contour = max(contours, key=cv2.contourArea)

        if cv2.contourArea(contour) < min_area:
            continue

        # Simplificar el polígono para reducir el número de puntos
        epsilon = 0.005 * cv2.arcLength(contour, True)
        contour = cv2.approxPolyDP(contour, epsilon, True)

        # Normalizar coordenadas al rango [0, 1]
        pts = contour.reshape(-1, 2).astype(float)
        pts[:, 0] /= W   # x normalizado
        pts[:, 1] /= H   # y normalizado

        # Formatear línea YOLO-seg:
        # class_id x1 y1 x2 y2 ... xn yn
        coords = ' '.join(f'{p:.6f}' for xy in pts for p in xy)
        lines.append(f'{yolo_id} {coords}')

    os.makedirs(os.path.dirname(txt_path), exist_ok=True)
    with open(txt_path, 'w') as f:
        f.write('\n'.join(lines))

    return len(lines)


def preparar_split(df_split: pd.DataFrame,
                   split_name: str,
                   yolo_root: str = 'yolo_dataset') -> None:
    """
    Copia imágenes y genera etiquetas .txt para un split del dataset.

    Estructura de salida:
        yolo_dataset/
            images/train/  images/val/  images/test/
            labels/train/  labels/val/  labels/test/
    """
    import shutil

    img_out = Path(yolo_root) / 'images' / split_name
    lbl_out = Path(yolo_root) / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    ok = 0
    for _, row in df_split.iterrows():
        img_src  = row['image']
        mask_src = row['mask']
        stem     = Path(img_src).stem

        # Copiar imagen
        shutil.copy(img_src, img_out / Path(img_src).name)

        # Generar etiqueta .txt
        txt_path = str(lbl_out / f'{stem}.txt')
        n = mask_png_a_yolo_txt(mask_src, txt_path)
        if n > 0:
            ok += 1

    print(f'[{split_name}] {ok}/{len(df_split)} imágenes con etiquetas generadas')


# ── Generar los tres splits ──────────────────────────────────────────
print('Convirtiendo máscaras PNG → etiquetas YOLO-seg...')
preparar_split(train_df, 'train')
preparar_split(val_df,   'val')
preparar_split(test_df,  'test')
print('\n✅ Conversión completada')

Convirtiendo máscaras PNG → etiquetas YOLO-seg...
[train] 173/174 imágenes con etiquetas generadas
[val] 37/37 imágenes con etiquetas generadas
[test] 38/38 imágenes con etiquetas generadas

✅ Conversión completada


## 📝 Celda 4: Archivo de configuración `dataset.yaml`

YOLOv8 requiere un archivo YAML que describe las rutas y las clases del dataset.
Es equivalente al argumento `classes=36` del constructor U-Net, pero declarado externamente.

In [4]:
import yaml

dataset_yaml = {
    'path' : str(Path('yolo_dataset').resolve()),
    'train': 'images/train',
    'val'  : 'images/val',
    'test' : 'images/test',
    'nc'   : N_CLASSES,           # 24 clases vertebrales
    'names': CLASS_NAMES
}

yaml_path = 'spine_dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, allow_unicode=True)

print('spine_dataset.yaml generado:')
print(open(yaml_path).read())

spine_dataset.yaml generado:
names:
- C1
- C2
- C3
- C4
- C5
- C6
- C7
- T1
- T2
- T3
- T4
- T5
- T6
- T7
- T8
- T9
- T10
- T11
- T12
- L1
- L2
- L3
- L4
- L5
nc: 24
path: /Users/fedys/Documents/MAIA/UNIANDES/semestre IV/Desarrollo de soluciones/proyecto_grado/eda/yolo_dataset
test: images/test
train: images/train
val: images/val



## 🧠 Celda 5: Fine-tuning de YOLOv8n-seg

### ¿Por qué `yolov8n-seg.pt`?

- **`n`** = nano: el modelo más ligero de la familia YOLOv8 (3.4M parámetros vs 25M+ de U-Net ResNet34).
  Ideal para entornos académicos con recursos limitados.
- **`-seg`**: variante de segmentación de instancias — añade una cabeza de máscara
  al detector base, generando un contorno por objeto detectado.
- **`.pt`**: pesos preentrenados en COCO (80 clases). Se hace **fine-tuning** sobre
  nuestras 24 clases vertebrales, aprovechando el transfer learning.

### Diferencia con U-Net en el entrenamiento
U-Net requería un loop manual con `optimizer.zero_grad()`, `loss.backward()`,
`optimizer.step()`. YOLOv8 encapsula todo en `model.train()`, que internamente
usa un loop similar pero con data augmentation automática, mixed precision,
cosine LR scheduler y early stopping integrados.

In [5]:
from ultralytics import YOLO

# ── Cargar YOLOv8n-seg con pesos preentrenados COCO ─────────────────
# Si yolov8n-seg.pt no está en disco, Ultralytics lo descarga automáticamente.
model = YOLO('yolov8n-seg.pt')

print('Arquitectura YOLOv8n-seg cargada')
print(f'Parámetros: {sum(p.numel() for p in model.model.parameters()):,}')

# ── Fine-tuning ──────────────────────────────────────────────────────
# Parámetros equivalentes a la configuración U-Net original:
#   epochs=10, batch=4, imgsz=512, optimizer=Adam, lr=1e-4
results = model.train(
    data      = 'spine_dataset.yaml',
    epochs    = 10,              # mismo que U-Net
    batch     = 4,               # mismo batch size
    imgsz     = 512,             # misma resolución
    optimizer = 'Adam',          # mismo optimizador
    lr0       = 1e-4,            # misma tasa de aprendizaje inicial
    lrf       = 0.01,            # LR final = lr0 * lrf (cosine decay)
    device    = device,
    project   = 'spine_yolo',    # carpeta de resultados
    name      = 'yolov8n_seg',
    exist_ok  = True,
    # ── Parámetros de segmentación ───────────────────────────────────
    task      = 'segment',
    # ── Augmentación (útil con dataset pequeño de 249 imágenes) ──────
    degrees   = 5.0,    # rotación leve (vértebras toleran pequeñas rotaciones)
    flipud    = 0.0,    # NO voltear verticalmente (vertebras tienen orientación)
    fliplr    = 0.5,    # voltear horizontal está bien (columna simétrica)
    mosaic    = 0.5,    # mosaico para aumentar contexto
    # ── Guardar el mejor modelo automáticamente ──────────────────────
    save      = True,
    patience  = 5,      # early stopping si no mejora en 5 épocas
)

print('\n✅ Entrenamiento completado')
print(f'Mejor modelo guardado en: spine_yolo/yolov8n_seg/weights/best.pt')

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-seg.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>


######################################################################## 100.0%


Arquitectura YOLOv8n-seg cargada
Parámetros: 3,409,968
Ultralytics 8.4.37 🚀 Python-3.13.7 torch-2.11.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=spine_dataset.yaml, degrees=5.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8n_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam,

## 📊 Celda 6: Evaluación en el conjunto de test

YOLOv8 reporta métricas estándar de segmentación de instancias:
- **mAP50-seg**: mean Average Precision con IoU ≥ 0.50 para las máscaras
- **mAP50-95-seg**: mAP promediado sobre umbrales IoU 0.50–0.95
- **Precision / Recall**: a nivel de detección de cada vértebra

In [8]:
# ── Cargar el mejor modelo guardado durante el entrenamiento ─────────
best_model = YOLO('runs/segment/spine_yolo/yolov8n_seg/weights/best.pt')

# ── Evaluar en test set ──────────────────────────────────────────────
metrics = best_model.val(
    data   = 'spine_dataset.yaml',
    split  = 'test',
    imgsz  = 512,
    batch  = 4,
    device = device,
)

print('\n── Métricas de segmentación (test set) ──')
print(f'mAP50-seg    : {metrics.seg.map50:.4f}')
print(f'mAP50-95-seg : {metrics.seg.map:.4f}')
print(f'Precision    : {metrics.seg.mp:.4f}')
print(f'Recall       : {metrics.seg.mr:.4f}')

Ultralytics 8.4.37 🚀 Python-3.13.7 torch-2.11.0 CPU (Apple M4)
YOLOv8n-seg summary (fused): 86 layers, 3,262,744 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 624.2±368.4 MB/s, size: 153.3 KB)
val: Scanning /Users/fedys/Documents/MAIA/UNIANDES/semestre IV/Desarrollo de soluciones/proyecto_grado/eda/yolo_dataset/labels/test... 38 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 3.1Kit/s 0.0s
val: New cache created: /Users/fedys/Documents/MAIA/UNIANDES/semestre IV/Desarrollo de soluciones/proyecto_grado/eda/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.9it/s 1.7s.2s
                   all         38        583     0.0276      0.404     0.0339     0.0196     0.0272      0.393     0.0327     0.0149
                    C1         33         33          0          0          0          0          0  

## 🔍 Celda 7: Inferencia y visualización de segmentación

A diferencia de U-Net (que devuelve un tensor `H×W` con el ID de clase por píxel),
YOLOv8 devuelve un objeto `Results` con:
- `boxes`: bounding boxes de cada vértebra detectada
- `masks`: máscara binaria por instancia (una por vértebra)
- `cls`: clase predicha para cada detección

In [13]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

# ── Seleccionar una imagen del test set ──────────────────────────────
test_img_path = test_df.sample(1, random_state=42).iloc[0]['image']
test_mask_path = test_df.sample(1, random_state=42).iloc[0]['mask']

# ── Inferencia ───────────────────────────────────────────────────────
results = best_model.predict(
    source = test_img_path,
    imgsz  = 512,
    conf   = 0.1,      # umbral de confianza mínimo
    iou    = 0.45,      # umbral NMS
    device = device,
    verbose= False,
)

result = results[0]

# ── Leer imagen y máscara real ───────────────────────────────────────
img_np    = cv2.cvtColor(cv2.imread(test_img_path), cv2.COLOR_BGR2RGB)
img_np    = cv2.resize(img_np, (512, 512))
mask_real = cv2.imread(test_mask_path, cv2.IMREAD_GRAYSCALE)
mask_real = cv2.resize(mask_real, (512, 512), interpolation=cv2.INTER_NEAREST)

# ── Construir mapa de segmentación predicho (como U-Net: clase por píxel) ─
# Combinar todas las máscaras de instancia en un único mapa semántico
# para visualización comparable con el mapa de U-Net.
pred_map = np.zeros((512, 512), dtype=np.uint8)

if result.masks is not None:
    masks_data = result.masks.data.cpu().numpy()   # (N, H, W)
    classes    = result.boxes.cls.cpu().numpy().astype(int)  # (N,)

    for mask_i, cls_i in zip(masks_data, classes):
        # Redimensionar máscara al tamaño de la imagen
        mask_resized = cv2.resize(
            (mask_i > 0.5).astype(np.uint8),
            (512, 512), interpolation=cv2.INTER_NEAREST
        )
        # pixel_value = yolo_class_id + 1 (para coincidir con máscara original)
        pred_map[mask_resized == 1] = cls_i + 1

    print(f'Vértebras detectadas: {len(classes)}')
    print('Clases:', [CLASS_NAMES[c] for c in classes])
else:
    print('⚠️  No se detectaron vértebras. Prueba a bajar el umbral conf.')

# ── Visualización comparativa ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_np)
axes[0].set_title('Imagen original')
axes[0].axis('off')

axes[1].imshow(mask_real, cmap='jet', vmin=0, vmax=N_CLASSES)
axes[1].set_title('Máscara real (GT)')
axes[1].axis('off')

axes[2].imshow(pred_map, cmap='jet', vmin=0, vmax=N_CLASSES)
axes[2].set_title('Predicción YOLOv8n-seg')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('comparacion_segmentacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: comparacion_segmentacion.png')

⚠️  No se detectaron vértebras. Prueba a bajar el umbral conf.


<Figure size 1500x500 with 3 Axes>

Figura guardada: comparacion_segmentacion.png


## 🎨 Celda 8: Visualización con nombres de vértebras

Equivalente a la Celda 5 de U-Net: dibuja el nombre de cada vértebra
en el centroide de su máscara predicha.

In [10]:
# ── Overlay con nombres de vértebras ────────────────────────────────
overlay_names = img_np.copy()

if result.masks is not None:
    masks_data = result.masks.data.cpu().numpy()
    classes    = result.boxes.cls.cpu().numpy().astype(int)
    confs      = result.boxes.conf.cpu().numpy()

    # Colores únicos por vértebra para diferenciarlas visualmente
    np.random.seed(0)
    colors = np.random.randint(80, 255, (N_CLASSES, 3), dtype=np.uint8)

    for mask_i, cls_i, conf_i in zip(masks_data, classes, confs):
        mask_resized = cv2.resize(
            (mask_i > 0.5).astype(np.uint8),
            (512, 512), interpolation=cv2.INTER_NEAREST
        )

        # Sombrear la región de la vértebra
        color = colors[cls_i % N_CLASSES].tolist()
        colored_mask = np.zeros_like(overlay_names)
        colored_mask[mask_resized == 1] = color
        overlay_names = cv2.addWeighted(overlay_names, 1.0, colored_mask, 0.4, 0)

        # Centroide para colocar el texto
        ys, xs = np.where(mask_resized == 1)
        if len(xs) == 0:
            continue
        cx, cy = int(np.mean(xs)), int(np.mean(ys))

        nombre = CLASS_NAMES[cls_i]
        label_text = f'{nombre} {conf_i:.2f}'

        # Fondo del texto para legibilidad
        (tw, th), _ = cv2.getTextSize(
            label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.35, 1
        )
        cv2.rectangle(overlay_names,
                      (cx - 2, cy - th - 2), (cx + tw + 2, cy + 2),
                      (0, 0, 0), -1)
        cv2.putText(
            overlay_names, label_text,
            (cx, cy),
            cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1, cv2.LINE_AA
        )

plt.figure(figsize=(6, 10))
plt.imshow(overlay_names)
plt.title('Segmentación YOLOv8n-seg con nombres')
plt.axis('off')
plt.tight_layout()
plt.savefig('segmentacion_con_nombres.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 600x1000 with 1 Axes>

## 📐 Celda 9: Orientación de vértebras mediante PCA

Idéntico al método del proyecto U-Net: se aplica PCA sobre los píxeles
de cada máscara de vértebra para estimar el eje principal (orientación).

La diferencia es que ahora los píxeles provienen de las **máscaras de instancia
de YOLOv8** en lugar del mapa semántico de U-Net.

In [11]:
angles  = []
centers = []
names   = []

if result.masks is not None:
    masks_data = result.masks.data.cpu().numpy()
    classes    = result.boxes.cls.cpu().numpy().astype(int)

    for mask_i, cls_i in zip(masks_data, classes):
        mask_resized = cv2.resize(
            (mask_i > 0.5).astype(np.uint8),
            (512, 512), interpolation=cv2.INTER_NEAREST
        )

        ys, xs = np.where(mask_resized == 1)

        if len(xs) < 20:   # región demasiado pequeña para PCA fiable
            continue

        # PCA sobre los píxeles de la vértebra (igual que U-Net)
        pts = np.column_stack((xs, ys)).astype(np.float32)
        mean, eigenvectors = cv2.PCACompute(pts, mean=None)

        # Primer eigenvector = dirección principal (eje mayor de la vértebra)
        vx, vy = eigenvectors[0]
        angle  = np.degrees(np.arctan2(vy, vx))

        angles.append(angle)
        centers.append((int(mean[0][0]), int(mean[0][1])))
        names.append(CLASS_NAMES[cls_i])

print(f'Vértebras con orientación calculada: {len(angles)}')
for n, a, c in zip(names, angles, centers):
    print(f'  {n:4s}: ángulo={a:7.2f}°  centro={c}')

Vértebras con orientación calculada: 0


## 📏 Celda 10: Estimación del ángulo de Cobb

El **ángulo de Cobb** es el estándar clínico para cuantificar la escoliosis.
Se mide entre las vértebras más inclinadas del tramo con mayor curvatura.

### Criterio diagnóstico
| Ángulo de Cobb | Clasificación |
|---|---|
| < 10° | Normal (variación postural) |
| 10° – 25° | Escoliosis leve |
| 25° – 40° | Escoliosis moderada |
| > 40° | Escoliosis severa (considerar cirugía) |

In [ ]:
if len(angles) < 2:
    print('⚠️  Se necesitan al menos 2 vértebras para calcular el ángulo de Cobb.')
else:
    angles_arr = np.array(angles)

    # Vértebra con mayor inclinación positiva y mayor inclinación negativa
    # (igual que la implementación U-Net original)
    upper_idx = int(np.argmax(angles_arr))
    lower_idx = int(np.argmin(angles_arr))

    upper_angle = angles_arr[upper_idx]
    lower_angle = angles_arr[lower_idx]

    cobb_angle = abs(upper_angle - lower_angle)

    # Clasificación clínica
    if cobb_angle < 10:
        diagnostico = 'Normal'
    elif cobb_angle < 25:
        diagnostico = 'Escoliosis leve'
    elif cobb_angle < 40:
        diagnostico = 'Escoliosis moderada'
    else:
        diagnostico = 'Escoliosis severa'

    print(f'Vértebra superior más inclinada : {names[upper_idx]} ({upper_angle:.2f}°)')
    print(f'Vértebra inferior más inclinada : {names[lower_idx]} ({lower_angle:.2f}°)')
    print(f'\nÁngulo de Cobb: {cobb_angle:.2f}°')
    print(f'Diagnóstico   : {diagnostico}')

## 🖼️ Celda 11: Visualización final del ángulo de Cobb

Dibuja las líneas de orientación sobre las dos vértebras extremas
(igual que la celda final del proyecto U-Net).

In [ ]:
if len(angles) >= 2:
    overlay_cobb = cv2.cvtColor(img_np.copy(), cv2.COLOR_RGB2BGR)

    for idx in [upper_idx, lower_idx]:
        x, y   = centers[idx]
        angle  = angles[idx]
        length = 100

        x2 = int(x + length * np.cos(np.radians(angle)))
        y2 = int(y + length * np.sin(np.radians(angle)))

        cv2.line(overlay_cobb, (x, y), (x2, y2), (0, 255, 0), 2)

        # Etiqueta con nombre de la vértebra
        cv2.putText(
            overlay_cobb, names[idx],
            (x + 5, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2, cv2.LINE_AA
        )

    plt.figure(figsize=(6, 10))
    plt.imshow(cv2.cvtColor(overlay_cobb, cv2.COLOR_BGR2RGB))
    plt.title(f'Ángulo de Cobb: {cobb_angle:.2f}°  —  {diagnostico}')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('cobb_angle_yolo.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: cobb_angle_yolo.png')

## 📦 Celda 12: Evaluación en lote del test set completo

Calcula el ángulo de Cobb para todas las imágenes del test set y compara
la predicción del diagnóstico (Normal / Escoliosis) con la etiqueta real.

In [ ]:
from tqdm import tqdm

COBB_THRESHOLD = 10.0   # ángulo mínimo para clasificar como escoliosis

resultados = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluando test set'):
    img_path  = row['image']
    label_real = row['label']   # 0=Normal, 1=Escoliosis

    # Inferencia YOLOv8
    res = best_model.predict(
        source=img_path, imgsz=512, conf=0.25, device=device, verbose=False
    )[0]

    cobb = None
    pred_label = 0   # por defecto: Normal

    if res.masks is not None and len(res.masks.data) >= 2:
        ang_list = []

        for mask_i in res.masks.data.cpu().numpy():
            mr = cv2.resize(
                (mask_i > 0.5).astype(np.uint8),
                (512, 512), interpolation=cv2.INTER_NEAREST
            )
            ys, xs = np.where(mr == 1)
            if len(xs) < 20:
                continue
            pts  = np.column_stack((xs, ys)).astype(np.float32)
            _, ev = cv2.PCACompute(pts, mean=None)
            ang_list.append(np.degrees(np.arctan2(ev[0][1], ev[0][0])))

        if len(ang_list) >= 2:
            ang_arr = np.array(ang_list)
            cobb = abs(ang_arr.max() - ang_arr.min())
            pred_label = 1 if cobb >= COBB_THRESHOLD else 0

    resultados.append({
        'imagen'     : img_path,
        'label_real' : label_real,
        'pred_label' : pred_label,
        'cobb_angle' : round(cobb, 2) if cobb is not None else None,
    })

results_df = pd.DataFrame(resultados)

# ── Métricas de clasificación ─────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

valid = results_df.dropna(subset=['cobb_angle'])

print('\n── Reporte de clasificación (Normal vs Escoliosis) ──')
print(classification_report(
    valid['label_real'], valid['pred_label'],
    target_names=['Normal', 'Escoliosis']
))

print('Matriz de confusión:')
print(confusion_matrix(valid['label_real'], valid['pred_label']))

print(f'\nImágenes sin detección suficiente: {results_df["cobb_angle"].isna().sum()}')
print(results_df[['cobb_angle', 'label_real', 'pred_label']].head(10))

## 📈 Celda 13: Curvas de entrenamiento

YOLOv8 guarda automáticamente los logs de entrenamiento en `spine_yolo/yolov8n_seg/results.csv`.

In [ ]:
results_csv = Path('spine_yolo/yolov8n_seg/results.csv')

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Curvas de entrenamiento YOLOv8n-seg', fontsize=13)

    # Pérdida de segmentación
    if 'train/seg_loss' in df_res.columns:
        axes[0].plot(df_res['epoch'], df_res['train/seg_loss'], label='Train')
        axes[0].plot(df_res['epoch'], df_res['val/seg_loss'],   label='Val')
        axes[0].set_title('Seg Loss')
        axes[0].legend()
        axes[0].set_xlabel('Época')

    # mAP50-seg
    if 'metrics/mAP50(M)' in df_res.columns:
        axes[1].plot(df_res['epoch'], df_res['metrics/mAP50(M)'])
        axes[1].set_title('mAP50-seg')
        axes[1].set_xlabel('Época')

    # mAP50-95-seg
    if 'metrics/mAP50-95(M)' in df_res.columns:
        axes[2].plot(df_res['epoch'], df_res['metrics/mAP50-95(M)'])
        axes[2].set_title('mAP50-95-seg')
        axes[2].set_xlabel('Época')

    plt.tight_layout()
    plt.savefig('curvas_entrenamiento_yolo.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Ejecuta el entrenamiento primero (Celda 5).')